In [1]:
import VAS
import VAS.tools as T
import pandas as pd
import VAS.metrics as MET
import copy

In [2]:
from VAS.vas_addons.pipeline.basic_pipeline import BasicPipeline
from VAS.vas_addons.tools.sort import SortBySelf
from VAS.vas_addons.tools.filter import FilterByValue
from VAS.vas_addons.metrics.group_wise_outcome_evaluation import GroupWiseOutcomeEvaluation

In [3]:
# import compass dataset
dataset_df = pd.read_csv("archive/compas-scores-raw.csv")

In [4]:
# lets look at first five rows to know what data we're dealing with
dataset_df.head()

,Person_ID,AssessmentID,Case_ID,Agency_Text,LastName,FirstName,MiddleName,Sex_Code_Text,Ethnic_Code_Text,DateOfBirth,...,RecSupervisionLevel,RecSupervisionLevelText,Scale_ID,DisplayText,RawScore,DecileScore,ScoreText,AssessmentType,IsCompleted,IsDeleted
0,50844,57167,51950,PRETRIAL,Fisher,Kevin,NaN,Male,Caucasian,12/05/92,...,1,Low,7,Risk of Violence,-2.08,4,Low,New,1,0
1,50844,57167,51950,PRETRIAL,Fisher,Kevin,NaN,Male,Caucasian,12/05/92,...,1,Low,8,Risk of Recidivism,-1.06,2,Low,New,1,0
2,50844,57167,51950,PRETRIAL,Fisher,Kevin,NaN,Male,Caucasian,12/05/92,...,1,Low,18,Risk of Failure to Appear,15.00,1,Low,New,1,0
3,50848,57174,51956,PRETRIAL,KENDALL,KEVIN,NaN,Male,Caucasian,09/16/84,...,1,Low,7,Risk of Violence,-2.84,2,Low,New,1,0
4,50848,57174,51956,PRETRIAL,KENDALL,KEVIN,NaN,Male,Caucasian,09/16/84,...,1,Low,8,Risk of Recidivism,-1.50,1,Low,New,1,0


In [5]:
## let's build a pipeline with a sort and two filter tools, and see if there are any group level (based on sex) discrepancies

# defining tools:
sort = SortBySelf()
filter_null = FilterByValue(func = lambda x: not pd.isna(x))
filter_probation = FilterByValue(func = lambda x: x=="Probation")

# defining metric
group_wise_outcome_eval = GroupWiseOutcomeEvaluation(col_with_groups="Sex_Code_Text", operation_on_col=lambda arr : sum(arr)/len(arr))

# define pipeline
pipeline = BasicPipeline()
# adding tools to the pipeline
pipeline.add_tool(sort, {"kwargs": {"col_to_sort_by": "DecileScore"}, "col_name": "sorted_by_DecileScore"})
pipeline.add_tool(filter_null, {"kwargs": {"col_to_filter_by": "MiddleName"}, "col_name": "filter_by_middle_name"})
pipeline.add_tool(filter_probation, {"kwargs": {"col_to_filter_by": "Agency_Text"}, "col_name": "filter_by_probation"})
# adding metrics to the pipeline
pipeline.add_metric(group_wise_outcome_eval, {"col_names": ["sorted_by_DecileScore", "filter_by_middle_name", "filter_by_probation"]})

In [6]:
# run the pipeline
res_df = pipeline.run(dataset=dataset_df)

In [7]:
# evaluate the metrics
pipeline.evaluate_metrics()

In [8]:
group_wise_evals = pipeline.evals

In [17]:
# lets run a dummy evaluation to see the effects of two filter tools (1. to filter out people with no middle names, and 2. to filter and keep only those on probation)

In [13]:
tool_name = [idx for idx, val in list(group_wise_evals.values())[0].items()][1:]
female_val = [val['Female'] for idx, val in list(group_wise_evals.values())[0].items()][1:]
male_val = [val['Male'] for idx, val in list(group_wise_evals.values())[0].items()][1:]

In [14]:
import plotly.express as px

In [15]:
data = {
    'Label': tool_name,
    'Female': female_val,
    'Male': male_val
}

df = pd.DataFrame(data)

# Create a bar plot
fig = px.bar(df, x='Label', y=['Female', 'Male'], barmode='group',
             labels={'Female': 'Female Group', 'Male': 'Male Group'},
             title='Comparison of Female and Male Groups for filter tools')

fig.update_layout(
    xaxis_title='Tool wise evaluation',
    yaxis_title='Values'       
)

In [ ]:
# as seen above, even though filtering by middle names in itself might not seem biased towards females or males, but we observe 
# that the filter removes more males as compared to females, i.e. a higher proportion of females have a middle name as compared to 
# men.
# For the probation filter, we see that males are more likely to have probation on their record, and a higher proportion of males
# stay in the dataset as compared to females